# B3a: Chronos Attribution Arm (Harmonic Sampling-Density and Noise Sweeps)

**Question:** is Panda's advantage on the Harmonic oscillator explained by a
Chronos-specific tokenization failure on smooth, densely-sampled periodic
signals, or does it reflect something else (general task difficulty, or a
Panda-side property)?

**Motivation:** Harmonic now has a confirmed, high-confidence advantage
(n=20, H=96, p<0.0001), but no established mechanism. Two threads tested
against it directly both point at unresolved anomalies rather than answers
(A3: the Koopman lift hurts Harmonic despite Harmonic being maximally
linearizable; A2a: Harmonic shows the largest relative degradation under
patch shuffling of any tested class). Given Panda's absolute MAE on Harmonic
is already near-ceiling while Chronos's is nearly 6x worse, this arm tests
the competing account flagged as far back as Experiment 19: "Chronos
specifically fails on sinusoidal inputs due to its discrete tokenisation
scheme" -- a Chronos-side story, not a Panda-side mechanism.

**Design correction made before building this** (see conversation): an
initial 3-axis plan (amplitude, frequency, sampling-density) was reduced to
2 genuinely independent axes.
- "Frequency" and "sampling density" are NOT independent at fixed
  CONTEXT_LEN=512: periods_in_window = 512 / samples_per_period, always.
  Testing both would have silently tested the same thing twice. Reduced to
  one axis: periods-in-window (equivalently, samples-per-period).
- "Amplitude" is removed by this project's own per-window instance
  normalisation (zero-mean/unit-variance) before either model sees the data
  -- a raw amplitude sweep under the standard harness is a guaranteed null
  by construction, not a real test. Dropped, replaced with noise level
  (signal-to-noise ratio survives normalisation meaningfully, and reuses
  Experiment 2's exact precedent noise levels rather than inventing new ones).

**Grid:**
- Axis 1 (periods-in-window): {1, 2, 4, 8, 16, 32}, via omega sweep at fixed
  dt=0.05. Log-spaced, matching this log's own convention (Burgers nu-sweep,
  Lorenz rho-sweep).
- Axis 2 (noise level, relative to signal std): {0, 0.01, 0.05, 0.10, 0.25,
  0.50}, verbatim from Experiment 2 (Double Pendulum graded noise).

Both models (Panda AND Chronos) run at every grid point, not Chronos alone,
so a change can be decomposed (Section 1.2 convention) into Chronos-specific
movement vs. general difficulty that Panda also shows. n_windows=20
throughout. H=96 (the established primary target for Harmonic), single pass.

**Pre-registered decision rule (fixed before running):**
- H1 (Chronos-specific tokenization brittleness): Chronos's MAE degrades
  specifically at high sampling density (small periods-in-window / large
  samples-per-period) while Panda's MAE stays comparatively flat across the
  same sweep.
- H2 (general difficulty, not Chronos-specific): both models' MAE move
  together in the same direction across the periods-in-window sweep.
- **Sharpest test, noise sweep:** if Chronos's relative skill IMPROVES with
  a small amount of added noise (counter-intuitive -- noise usually hurts),
  that is specific, strong evidence for smooth-signal tokenization brittleness,
  since noise would break up the exact-repetition pattern a naive discretizer
  chokes on. If Chronos simply degrades monotonically with noise like
  everything else, this prediction fails and H1 loses support.


In [1]:
# ============================================================
# CELL 1 -- IMPORTS + MODEL LOADING (published checkpoints)
# Verbatim from fixed_experiments.ipynb, Cell 0 (same pattern used throughout
# this project's local notebooks).
# ============================================================
import numpy as np
import pandas as pd
import torch
from scipy.stats import wilcoxon
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

import sys
sys.path.insert(0, './panda')  # local run: panda repo cloned in the working directory

from panda.patchtst.pipeline import PatchTSTPipeline
from chronos import ChronosPipeline

panda_model = PatchTSTPipeline.from_pretrained(
    mode="predict",
    pretrain_path="GilpinLab/panda",
    device_map=device,
)

chronos_model = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small",
    device_map=device,
    torch_dtype=torch.bfloat16,
)

print("Models loaded.")


Device: cpu
Models loaded.


In [2]:
# ============================================================
# CELL 2 -- EVALUATION HARNESS
# Verbatim from fixed_experiments.ipynb / harmonic_n20_confirmatory.ipynb.
# ============================================================
def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))

def instance_norm_window(x_CT):
    """x_CT: (C, T). Normalise per channel using this window only."""
    mu  = x_CT.mean(axis=1, keepdims=True)
    std = x_CT.std( axis=1, keepdims=True) + 1e-8
    return (x_CT - mu) / std, mu, std

CONTEXT_LEN = 512

def panda_forecast(context_np, horizon):
    """context_np: (C, T) normalised. Returns (C, horizon)."""
    TRAIN_H   = 128
    remaining = horizon
    ctx       = context_np.copy()
    preds     = []
    while remaining > 0:
        h         = min(TRAIN_H, remaining)
        context_t = torch.tensor(ctx.T, dtype=torch.float32)
        with torch.no_grad():
            pred = panda_model.predict(
                context_t, h,
                limit_prediction_length=False,
                sliding_context=True,
            )
        p = pred.squeeze().cpu().numpy()
        if p.ndim == 1:
            p = p[:, None]
        if p.shape[0] != context_np.shape[0]:
            p = p.T
        preds.append(p[:, :h])
        ctx       = np.concatenate([ctx[:, h:], p[:, :h]], axis=1)
        remaining -= h
    return np.concatenate(preds, axis=1)  # (C, horizon)

def chronos_forecast(context_np, horizon):
    """Batched -- all channels in one call."""
    ctx = torch.tensor(context_np, dtype=torch.float32)
    with torch.no_grad():
        out = chronos_model.predict(
            ctx, prediction_length=horizon, num_samples=1
        )
    return out[:, 0, :].cpu().numpy()  # (C, horizon)

def evaluate(data_CT, horizon, n_windows=20, label="",
             fn_a=None, fn_b=None,
             name_a="panda", name_b="chronos"):
    """data_CT: (C, T) RAW. Normalises each window independently."""
    if fn_a is None: fn_a = panda_forecast
    if fn_b is None: fn_b = chronos_forecast

    C, T      = data_CT.shape
    max_start = T - CONTEXT_LEN - horizon
    if max_start <= 0:
        print(f"  [SKIP] {label}: T={T} too short")
        return None

    starts = np.linspace(0, max_start, n_windows, dtype=int)
    mae_a, mae_b = [], []

    for s in starts:
        ctx_raw = data_CT[:, s : s + CONTEXT_LEN]
        tgt_raw = data_CT[:, s + CONTEXT_LEN : s + CONTEXT_LEN + horizon]
        ctx_norm, mu, std = instance_norm_window(ctx_raw)
        tgt_norm = (tgt_raw - mu) / std

        mae_a.append(mae(tgt_norm, fn_a(ctx_norm, horizon)))
        mae_b.append(mae(tgt_norm, fn_b(ctx_norm, horizon)))

    diff = np.array(mae_b) - np.array(mae_a)
    try:
        if np.any(diff != 0):
            _, pval = wilcoxon(diff, alternative="greater")
        else:
            pval = 1.0
    except Exception:
        pval = np.nan

    adv = np.median(mae_b) - np.median(mae_a)
    rel_skill = np.median(mae_b) / np.median(mae_a) if np.median(mae_a) > 0 else np.nan

    result = {
        "label"         : label,
        "horizon"       : horizon,
        f"{name_a}_mae" : np.median(mae_a),
        f"{name_b}_mae" : np.median(mae_b),
        "advantage_mae" : adv,
        "rel_skill"     : rel_skill,
        "wilcoxon_p"    : pval,
    }
    return result

print("Helpers defined.")


Helpers defined.


In [3]:
# ============================================================
# CELL 3 -- HARMONIC GENERATOR, PARAMETRISED BY periods_in_window AND noise_frac
# Base generator verbatim from eval-nb.ipynb / harmonic_n20_confirmatory.ipynb
# (n_steps=4000, dt=0.05, seed=42 base convention), extended with an explicit
# omega control (for periods-in-window) and additive Gaussian noise (for the
# noise sweep). At periods_in_window=4.07-ish (omega=1.0, dt=0.05, this
# project's established default) and noise_frac=0, this reduces EXACTLY to
# the already-confirmed Harmonic protocol.
# ============================================================
SEED = 42
BASE_DT = 0.05
N_STEPS = 4000
TRANSIENT_SKIP = 500

def simulate_harmonic(omega=1.0, dt=BASE_DT, n_steps=N_STEPS, seed=SEED, noise_frac=0.0):
    rng = np.random.default_rng(seed)
    x, v = float(rng.standard_normal()), float(rng.standard_normal())
    traj = []
    for _ in range(n_steps):
        traj.append(x)
        x_new = x + v * dt
        v_new = v - omega**2 * x * dt
        x, v  = x_new, v_new
    traj = np.array(traj, dtype=np.float32)
    if noise_frac > 0:
        sig_std = traj[TRANSIENT_SKIP:].std()
        traj = traj + rng.normal(0, noise_frac * sig_std, size=traj.shape).astype(np.float32)
    return traj

def omega_for_periods_in_window(periods_in_window, dt=BASE_DT, context_len=CONTEXT_LEN):
    """periods_in_window = context_len * omega * dt / (2*pi)  ->  solve for omega."""
    return periods_in_window * 2 * np.pi / (context_len * dt)

def load_harmonic(omega=1.0, noise_frac=0.0):
    series = simulate_harmonic(omega=omega, dt=BASE_DT, n_steps=N_STEPS,
                                seed=SEED, noise_frac=noise_frac)
    return series[TRANSIENT_SKIP:][None, :]

# Sanity check: baseline (omega=1.0, noise=0) should match the already-confirmed protocol.
_baseline = load_harmonic(omega=1.0, noise_frac=0.0)
print(f"Baseline sanity check: shape={_baseline.shape}, "
      f"periods_in_window ~= {CONTEXT_LEN * 1.0 * BASE_DT / (2*np.pi):.2f} "
      f"(should be close to the ~4.07 seen in the confirmatory rerun)")


Baseline sanity check: shape=(1, 3500), periods_in_window ~= 4.07 (should be close to the ~4.07 seen in the confirmatory rerun)


In [4]:
# ============================================================
# CELL 4 -- SWEEP 1: PERIODS-IN-WINDOW (samples-per-period, reciprocal axis)
# ============================================================
PERIODS_IN_WINDOW_GRID = [1, 2, 4, 8, 16, 32]
H = 96  # established primary target, single pass since H <= 128
N_WINDOWS = 20

sweep1_results = []
print("Sweep 1: periods-in-window (H=96, n_windows=20)")
print()
for piw in PERIODS_IN_WINDOW_GRID:
    omega = omega_for_periods_in_window(piw)
    samples_per_period = CONTEXT_LEN / piw
    data = load_harmonic(omega=omega, noise_frac=0.0)
    r = evaluate(data, H, n_windows=N_WINDOWS, label=f"piw={piw}")
    if r:
        r["periods_in_window"] = piw
        r["samples_per_period"] = samples_per_period
        sweep1_results.append(r)
        print(f"  periods_in_window={piw:3d}  samples_per_period={samples_per_period:6.1f}  "
              f"panda={r['panda_mae']:.4f}  chronos={r['chronos_mae']:.4f}  "
              f"adv={r['advantage_mae']:+.4f}  rel_skill={r['rel_skill']:.3f}  p={r['wilcoxon_p']:.4f}")

df_sweep1 = pd.DataFrame(sweep1_results)
df_sweep1.to_csv("b3a_periods_in_window_sweep.csv", index=False)
print()
print("Saved b3a_periods_in_window_sweep.csv")


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 


Sweep 1: periods-in-window (H=96, n_windows=20)



We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  periods_in_window=  1  samples_per_period= 512.0  panda=0.0174  chronos=0.2072  adv=+0.1898  rel_skill=11.920  p=0.0000


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

...

  periods_in_window=  8  samples_per_period=  64.0  panda=0.2783  chronos=0.6707  adv=+0.3924  rel_skill=2.410  p=0.0000


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  periods_in_window= 16  samples_per_period=  32.0  panda=0.0002  chronos=0.0032  adv=+0.0030  rel_skill=16.745  p=0.4698


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  periods_in_window= 32  samples_per_period=  16.0  panda=nan  chronos=nan  adv=+nan  rel_skill=nan  p=nan

Saved b3a_periods_in_window_sweep.csv


In [5]:
# ============================================================
# CELL 5 -- SWEEP 2: NOISE LEVEL (baseline omega=1.0, ~4 periods in window)
# Noise levels verbatim from Experiment 2 (Double Pendulum graded noise).
# ============================================================
NOISE_GRID = [0.0, 0.01, 0.05, 0.10, 0.25, 0.50]

sweep2_results = []
print("Sweep 2: noise level (H=96, n_windows=20, baseline omega=1.0)")
print()
for nf in NOISE_GRID:
    data = load_harmonic(omega=1.0, noise_frac=nf)
    r = evaluate(data, H, n_windows=N_WINDOWS, label=f"noise={nf}")
    if r:
        r["noise_frac"] = nf
        sweep2_results.append(r)
        print(f"  noise_frac={nf:.2f}  panda={r['panda_mae']:.4f}  chronos={r['chronos_mae']:.4f}  "
              f"adv={r['advantage_mae']:+.4f}  rel_skill={r['rel_skill']:.3f}  p={r['wilcoxon_p']:.4f}")

df_sweep2 = pd.DataFrame(sweep2_results)
df_sweep2.to_csv("b3a_noise_sweep.csv", index=False)
print()
print("Saved b3a_noise_sweep.csv")


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 


Sweep 2: noise level (H=96, n_windows=20, baseline omega=1.0)



We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  noise_frac=0.00  panda=0.0673  chronos=0.4787  adv=+0.4114  rel_skill=7.114  p=0.0000


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

...

  noise_frac=0.10  panda=0.3530  chronos=0.7587  adv=+0.4057  rel_skill=2.149  p=0.0000


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  noise_frac=0.25  panda=0.5975  chronos=0.7578  adv=+0.1604  rel_skill=1.268  p=0.0000


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  noise_frac=0.50  panda=0.7296  chronos=0.8103  adv=+0.0807  rel_skill=1.111  p=0.0010

Saved b3a_noise_sweep.csv


In [6]:
# ============================================================
# CELL 6 -- PRE-REGISTERED DECISION RULE EVALUATION
# ============================================================
print("="*70)
print("SWEEP 1 (periods-in-window) interpretation")
print("="*70)
# High sampling density = small periods_in_window / large samples_per_period.
# Compare Chronos MAE and Panda MAE at the two densest points vs the two
# sparsest points.
df1 = df_sweep1.sort_values("periods_in_window")
dense = df1.iloc[:2]   # smallest periods_in_window = densest sampling
sparse = df1.iloc[-2:] # largest periods_in_window = sparsest sampling

chronos_dense_mean = dense["chronos_mae"].mean()
chronos_sparse_mean = sparse["chronos_mae"].mean()
panda_dense_mean = dense["panda_mae"].mean()
panda_sparse_mean = sparse["panda_mae"].mean()

chronos_pct_change = 100 * (chronos_sparse_mean - chronos_dense_mean) / chronos_dense_mean
panda_pct_change = 100 * (panda_sparse_mean - panda_dense_mean) / panda_dense_mean

print(f"Chronos MAE: dense(mean)={chronos_dense_mean:.4f} -> sparse(mean)={chronos_sparse_mean:.4f} "
      f"({chronos_pct_change:+.1f}%)")
print(f"Panda MAE:   dense(mean)={panda_dense_mean:.4f} -> sparse(mean)={panda_sparse_mean:.4f} "
      f"({panda_pct_change:+.1f}%)")
print()
big_gap = abs(chronos_pct_change) > 2 * abs(panda_pct_change) and abs(chronos_pct_change) > 20
same_direction_similar_size = (
    np.sign(chronos_pct_change) == np.sign(panda_pct_change)
    and abs(chronos_pct_change - panda_pct_change) < 0.5 * max(abs(chronos_pct_change), abs(panda_pct_change), 1e-9)
)
if big_gap:
    print("-> H1 SUPPORTED: Chronos moves substantially more than Panda across the "
          "sampling-density sweep (Chronos-specific effect).")
elif same_direction_similar_size:
    print("-> H2 SUPPORTED: both models move together in similar proportion "
          "(general difficulty, not Chronos-specific).")
else:
    print("-> MIXED / inconclusive by this rule -- inspect the full table below directly.")

print()
print(df1[["periods_in_window", "samples_per_period", "panda_mae", "chronos_mae",
           "rel_skill", "wilcoxon_p"]].round(4).to_string(index=False))

print()
print("="*70)
print("SWEEP 2 (noise level) interpretation -- the sharpest test")
print("="*70)
df2 = df_sweep2.sort_values("noise_frac")
rel_skill_at_zero = df2[df2["noise_frac"] == 0.0]["rel_skill"].iloc[0]
rel_skill_at_small_noise = df2[df2["noise_frac"] == 0.01]["rel_skill"].iloc[0]

print(f"Relative skill (Chronos_MAE/Panda_MAE) at noise=0:    {rel_skill_at_zero:.3f}")
print(f"Relative skill (Chronos_MAE/Panda_MAE) at noise=0.01: {rel_skill_at_small_noise:.3f}")
print("(Higher relative skill = Panda comparatively better; a DROP toward 1.0 "
      "means Chronos is catching up / improving relatively with a little noise.)")
print()
if rel_skill_at_small_noise < rel_skill_at_zero:
    print("-> SHARP PREDICTION CONFIRMED: relative skill drops with a small amount of "
          "noise -- Chronos improves RELATIVELY, consistent with smooth-signal "
          "tokenization brittleness that noise disrupts. Strong, specific support for H1.")
else:
    print("-> SHARP PREDICTION FAILS: relative skill does not drop with small noise. "
          "Chronos does not show the predicted noise-relief pattern -- H1 loses "
          "its most specific piece of support.")

print()
print(df2[["noise_frac", "panda_mae", "chronos_mae", "rel_skill", "wilcoxon_p"]]
      .round(4).to_string(index=False))


SWEEP 1 (periods-in-window) interpretation
Chronos MAE: dense(mean)=0.2529 -> sparse(mean)=0.0032 (-98.7%)
Panda MAE:   dense(mean)=0.0186 -> sparse(mean)=0.0002 (-99.0%)

-> H2 SUPPORTED: both models move together in similar proportion (general difficulty, not Chronos-specific).

 periods_in_window  samples_per_period  panda_mae  chronos_mae  rel_skill  wilcoxon_p
                 1               512.0     0.0174       0.2072    11.9201      0.0000
                 2               256.0     0.0199       0.2987    15.0079      0.0000
                 4               128.0     0.0641       0.3871     6.0380      0.0000
                 8                64.0     0.2783       0.6707     2.4097      0.0000
                16                32.0     0.0002       0.0032    16.7447      0.4698
                32                16.0        NaN          NaN        NaN         NaN

SWEEP 2 (noise level) interpretation -- the sharpest test
Relative skill (Chronos_MAE/Panda_MAE) at noise=0:    7.1